# 01 — Modelo Baseline para Classificação com Iris Dataset

Este notebook é uma versão adaptada do `01_baseline_classificacao.ipynb` para o **Iris Dataset**.

O objetivo é prever a espécie de uma flor Iris a partir de quatro variáveis preditoras:

- comprimento da sépala;
- largura da sépala;
- comprimento da pétala;
- largura da pétala.

Classes possíveis:

- `setosa`;
- `versicolor`;
- `virginica`.

## Premissa

Este exemplo usa um dataset já preparado, disponível diretamente no `scikit-learn`.

Ou seja:

- não há valores nulos;
- as variáveis preditoras já são numéricas;
- a variável-alvo já está definida;
- o problema é de classificação multiclasse.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler

from sklearn.dummy import DummyClassifier
from sklearn.tree import DecisionTreeClassifier, plot_tree

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

## 1. Carregar o dataset


Neste notebook, vamos carregar o Iris Dataset diretamente do `scikit-learn`.

O dataset possui 150 registros e 3 classes de flores Iris.

In [ ]:
iris = load_iris()

X = pd.DataFrame(
    iris.data,
    columns=iris.feature_names
)

y = pd.Series(
    iris.target,
    name="target"
)

df = X.copy()
df["target"] = y
df["species"] = df["target"].map({
    i: name for i, name in enumerate(iris.target_names)
})

df.head()

## 2. Visualização inicial dos dados


Mesmo com os dados já preparados, é importante entender a estrutura do dataset.

In [ ]:
print("Linhas e colunas:", df.shape)

df.info()

In [ ]:
df.describe(include="all")

In [ ]:
df.isnull().sum()

## 3. Visualização gráfica

Os gráficos ajudam a entender distribuição, dispersão e possíveis diferenças entre as variáveis.

In [ ]:
X.hist(figsize=(12, 8))
plt.tight_layout()
plt.show()

In [ ]:
X.plot(
    kind="box",
    figsize=(12, 6),
    rot=45
)

plt.title("Boxplot das variáveis numéricas do Iris Dataset")
plt.show()

## 4. Visualização de relação entre variáveis

Uma visualização simples e bastante didática é comparar:

- comprimento da pétala;
- largura da pétala.

Essas duas variáveis costumam ajudar bastante na separação das espécies.

In [ ]:
plt.figure(figsize=(8, 6))

for target_id, species_name in enumerate(iris.target_names):
    subset = df[df["target"] == target_id]

    plt.scatter(
        subset["petal length (cm)"],
        subset["petal width (cm)"],
        label=species_name
    )

plt.xlabel("Petal length (cm)")
plt.ylabel("Petal width (cm)")
plt.title("Relação entre comprimento e largura da pétala")
plt.legend()
plt.show()

## 5. Matriz de correlação

A matriz de correlação ajuda a identificar relações lineares entre variáveis numéricas.

In [ ]:
correlacao = X.corr()

plt.figure(figsize=(10, 6))
plt.imshow(correlacao, aspect="auto")
plt.colorbar()
plt.xticks(range(len(correlacao.columns)), correlacao.columns, rotation=90)
plt.yticks(range(len(correlacao.columns)), correlacao.columns)
plt.title("Matriz de correlação")
plt.show()

correlacao

## 6. Separar variáveis preditoras e variável-alvo

A variável-alvo é a coluna que o modelo deve aprender a prever.

Neste caso:

- `X`: medidas da flor;
- `y`: espécie da flor codificada numericamente.

In [ ]:
target = "target"

X = df[iris.feature_names]
y = df[target]

print("Variáveis preditoras:")
print(X.head())

print("\nVariável-alvo:")
print(y.head())

print("\nMapeamento das classes:")
for i, name in enumerate(iris.target_names):
    print(i, "=", name)

## 7. Analisar distribuição das classes

Esta etapa é importante para identificar se o problema está balanceado ou desbalanceado.

O Iris Dataset é bem balanceado: possui 50 amostras de cada classe.

In [ ]:
print("Contagem por classe:")
print(y.value_counts().sort_index())

print("\nProporção por classe:")
print(y.value_counts(normalize=True).sort_index())

print("\nContagem por espécie:")
print(df["species"].value_counts())

## 8. Dividir treino e teste

Usamos `stratify=y` para manter a proporção das classes no treino e no teste.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Treino:", X_train.shape)
print("Teste:", X_test.shape)

## 9. Normalização

A normalização é importante para modelos sensíveis à escala, como:

- Regressão Logística;
- KNN;
- SVM;
- Redes Neurais.

Para árvores de decisão, normalização normalmente não é obrigatória.

Regra importante:

- `fit_transform` apenas no treino;
- `transform` no teste.

In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

### Outras opções de normalização

Use uma das opções abaixo conforme o objetivo didático.

In [ ]:
# MinMaxScaler: coloca os valores normalmente entre 0 e 1
# scaler = MinMaxScaler()

# RobustScaler: mais robusto a outliers
# scaler = RobustScaler()

## 10. Baseline ingênuo

O `DummyClassifier` cria um modelo extremamente simples.

Ele serve como referência mínima para comparar o modelo real.

A estratégia `most_frequent` sempre prevê a classe mais frequente.

Como o Iris Dataset é balanceado, esse baseline tende a acertar aproximadamente 33%.

In [ ]:
dummy_model = DummyClassifier(strategy="most_frequent")

dummy_model.fit(X_train, y_train)

y_pred_dummy = dummy_model.predict(X_test)

print("Acurácia do baseline ingênuo:", accuracy_score(y_test, y_pred_dummy))

## 11. Modelo baseline real

Para fins didáticos, a árvore de decisão é uma boa escolha inicial.

Hiperparâmetros usados:

- `max_depth`: profundidade máxima da árvore;
- `min_samples_split`: mínimo de amostras para dividir um nó;
- `random_state`: garante reprodutibilidade.

In [ ]:
model = DecisionTreeClassifier(
    max_depth=3,
    min_samples_split=5,
    random_state=42
)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

## 12. Avaliação do modelo de classificação

Métricas principais:

- **Acurácia**: percentual geral de acertos.
- **Precisão**: das previsões de uma classe, quantas estavam corretas.
- **Recall**: dos exemplos reais de uma classe, quantos foram encontrados.
- **F1-score**: equilíbrio entre precisão e recall.
- **Matriz de confusão**: mostra onde o modelo acertou e errou.

In [ ]:
print("=== Baseline Ingênuo ===")
print("Acurácia:", accuracy_score(y_test, y_pred_dummy))

print("\n=== Modelo Baseline Real ===")
print("Acurácia:", accuracy_score(y_test, y_pred))
print("Precisão:", precision_score(y_test, y_pred, average="weighted", zero_division=0))
print("Recall:", recall_score(y_test, y_pred, average="weighted", zero_division=0))
print("F1-score:", f1_score(y_test, y_pred, average="weighted", zero_division=0))

print("\nRelatório de classificação:")
print(
    classification_report(
        y_test,
        y_pred,
        target_names=iris.target_names,
        zero_division=0
    )
)

## 13. Matriz de confusão

A matriz de confusão compara os valores reais com os valores previstos.

Regra didática:

- a diagonal principal representa os acertos;
- os valores fora da diagonal representam os erros.

Para o Iris Dataset, as classes são:

- `setosa`;
- `versicolor`;
- `virginica`.

In [ ]:
cm = confusion_matrix(y_test, y_pred)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=iris.target_names
)

disp.plot()
plt.title("Matriz de Confusão — Iris Dataset")
plt.show()

## 14. Visualização da árvore de decisão

Uma vantagem didática da árvore de decisão é que podemos visualizar as regras aprendidas pelo modelo.

In [ ]:
plt.figure(figsize=(18, 8))

plot_tree(
    model,
    feature_names=iris.feature_names,
    class_names=iris.target_names,
    filled=True,
    rounded=True
)

plt.title("Árvore de Decisão — Iris Dataset")
plt.show()

## 15. Percentual de acerto

Em classificação, o percentual de acerto corresponde à acurácia.

In [ ]:
percentual_acerto = accuracy_score(y_test, y_pred) * 100

print(f"Percentual de acerto no teste: {percentual_acerto:.2f}%")

## 16. Predição com uma amostra real do teste

Esta etapa ajuda o aluno a entender como o modelo usa um registro para gerar uma previsão.

In [ ]:
vetor_teste = X_test.iloc[[0]]

predicao = model.predict(vetor_teste)
valor_real = y_test.iloc[0]

print("Vetor de teste:")
print(vetor_teste)

print("\nPredição:", predicao[0], "-", iris.target_names[predicao[0]])
print("Valor real:", valor_real, "-", iris.target_names[valor_real])
print("Acertou?", predicao[0] == valor_real)

## 17. Predição com vetor manual

Agora vamos criar uma nova flor manualmente.

A ordem das variáveis precisa seguir exatamente as colunas usadas no treinamento:

1. `sepal length (cm)`
2. `sepal width (cm)`
3. `petal length (cm)`
4. `petal width (cm)`

In [ ]:
nova_flor = pd.DataFrame([{
    "sepal length (cm)": 5.1,
    "sepal width (cm)": 3.5,
    "petal length (cm)": 1.4,
    "petal width (cm)": 0.2
}])

predicao_nova_flor = model.predict(nova_flor)

print("Nova flor:")
print(nova_flor)

print(
    "\nClasse prevista:",
    predicao_nova_flor[0],
    "-",
    iris.target_names[predicao_nova_flor[0]]
)

## 18. Probabilidades por classe

Alguns modelos de classificação conseguem retornar uma probabilidade estimada para cada classe.

No caso da árvore de decisão, podemos usar `predict_proba`.

In [ ]:
probabilidades = model.predict_proba(nova_flor)

df_probabilidades = pd.DataFrame(
    probabilidades,
    columns=iris.target_names
)

df_probabilidades

## Sugestões didáticas

1. Mostre o `DummyClassifier` antes do modelo real.
2. Explique que o Iris é um problema de classificação multiclasse.
3. Use a matriz de confusão para mostrar quais espécies foram confundidas.
4. Use a árvore de decisão para explicar regras simples.
5. Compare os resultados com e sem normalização usando outro algoritmo, como `LogisticRegression`.
6. Mostre que `setosa` costuma ser mais fácil de separar das demais espécies.

## Desafio para os alunos

1. Troque o modelo `DecisionTreeClassifier` por `LogisticRegression`.
2. Use os dados normalizados: `X_train_scaled` e `X_test_scaled`.
3. Compare as métricas dos dois modelos.
4. Verifique se a matriz de confusão muda.
5. Altere os valores da `nova_flor` e observe a classe prevista.

In [ ]:
# Exemplo opcional para testar LogisticRegression

# from sklearn.linear_model import LogisticRegression

# logistic_model = LogisticRegression(max_iter=1000, random_state=42)
# logistic_model.fit(X_train_scaled, y_train)

# y_pred_logistic = logistic_model.predict(X_test_scaled)

# print(classification_report(
#     y_test,
#     y_pred_logistic,
#     target_names=iris.target_names,
#     zero_division=0
# ))